# 03. Movie Metadata Unification

이 노트북의 목적은 `Movie_Master_v1.csv`를 기준 테이블로 두고 Wavve 크롤링 메타데이터와 KOBIS 보완 메타데이터를 통합하는 것이다.

이번 수정본의 핵심 정책은 다음과 같다.

1. 입력 데이터는 repo root의 `_data`에서 읽는다.
2. v1 데이터 폴더명은 하드코딩하지 않고, `Movie_Master_v1.csv`와 `View_History_v1.csv`가 실제로 있는 폴더를 탐색한다.
3. `movie_5171.csv`는 `_data/02_interim/260506_movie(5171)/movie_5171.csv`에 있는 보조 메타데이터로 사용한다.
4. 산출 데이터는 `_data`에 쓰지 않고, `park.ingyeom/reports/data/03_movie_metadata_unification/`에 저장한다.
5. 검산표는 `park.ingyeom/reports/tables/`에 저장한다.

다운스트림 05번 콘텐츠 피처 생성에서는 `movie_metadata_unified_v2.csv`를 기본 입력으로 사용한다.


In [20]:
from pathlib import Path
import json
import re
from difflib import SequenceMatcher
from collections import Counter

import numpy as np
import pandas as pd

In [21]:
def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root under the 01/02 path policy.

    Correct local structure:
    C:/Code/ott-churn-prediction
    ├─ .git
    ├─ _data
    └─ park.ingyeom
       └─ notebooks

    The returned PROJECT_ROOT must be C:/Code/ott-churn-prediction, not park.ingyeom.
    """
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]

    for candidate in candidates:
        if (candidate / '.git').exists() and (candidate / '_data').exists():
            return candidate

    for candidate in candidates:
        if candidate.name == 'park.ingyeom' and (candidate.parent / '_data').exists():
            return candidate.parent

    for candidate in candidates:
        if (candidate / '_data' / '01_raw').exists() and (candidate / '_data' / '02_interim').exists():
            return candidate

    # Fallback for uploaded-file execution checks outside the local repository.
    return start


def find_dir_containing_files(base_dir: Path, required_files: list[str], preferred_name_contains: list[str] | None = None) -> Path:
    """Find a directory under base_dir that contains all required files.

    This avoids hard-coding the fragile v1 folder name. In the local repo, the observed
    folder is _data/02_interim/260430_membership_v1(이상치, 이름변경), while older notes
    used a different spelling. The actual files are the source of truth.
    """
    preferred_name_contains = preferred_name_contains or []
    if not base_dir.exists():
        raise FileNotFoundError(f'Base directory does not exist: {base_dir}')

    candidates = []
    for d in [base_dir, *[p for p in base_dir.rglob('*') if p.is_dir()]]:
        if all((d / file_name).exists() for file_name in required_files):
            score = sum(token in d.name for token in preferred_name_contains)
            candidates.append((score, len(d.parts), d))

    if not candidates:
        checked = ', '.join(required_files)
        raise FileNotFoundError(f'Could not find a directory under {base_dir} containing all files: {checked}')

    candidates.sort(key=lambda x: (-x[0], x[1], str(x[2])))
    return candidates[0][2]


def find_optional_file(base_dir: Path, file_name: str, preferred_parent_contains: list[str] | None = None) -> Path | None:
    """Find an optional file under base_dir, preferring directories whose names match tokens."""
    preferred_parent_contains = preferred_parent_contains or []
    if not base_dir.exists():
        return None
    matches = list(base_dir.rglob(file_name))
    if not matches:
        return None
    ranked = []
    for p in matches:
        score = sum(token in p.parent.name for token in preferred_parent_contains)
        ranked.append((score, len(p.parts), p))
    ranked.sort(key=lambda x: (-x[0], x[1], str(x[2])))
    return ranked[0][2]


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / '_data'
RAW_DIR = DATA_ROOT / '01_raw'
INTERIM_DIR = DATA_ROOT / '02_interim'

# Do not hard-code the v1 directory name. Detect it from the actual required v1 files.
V1_DATA_DIR = find_dir_containing_files(
    INTERIM_DIR,
    required_files=['Movie_Master_v1.csv', 'View_History_v1.csv'],
    preferred_name_contains=['260430', 'membership_v1'],
)

PATH_MOVIE_5171_CANDIDATE = find_optional_file(
    INTERIM_DIR,
    'movie_5171.csv',
    preferred_parent_contains=['260506_movie(5171)', '5171'],
)
MOVIE_5171_DIR = PATH_MOVIE_5171_CANDIDATE.parent if PATH_MOVIE_5171_CANDIDATE is not None else INTERIM_DIR / '260506_movie(5171)'

WORK_ROOT = PROJECT_ROOT / 'park.ingyeom' if (PROJECT_ROOT / 'park.ingyeom').exists() else PROJECT_ROOT
REPORTS_DIR = WORK_ROOT / 'reports'
DATA_OUTPUT_DIR = REPORTS_DIR / 'data' / '03_movie_metadata_unification'
OUTPUT_DIR = DATA_OUTPUT_DIR
TABLES_DIR = REPORTS_DIR / 'tables' / '03_movie_metadata_unification'

DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Create only personal output/report directories.
# Do not create or mutate repository-level _data directories here.
for d in [DATA_OUTPUT_DIR, TABLES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('cwd:', Path.cwd())
print('PROJECT_ROOT:', PROJECT_ROOT)
print('DATA_ROOT:', DATA_ROOT)
print('RAW_DIR:', RAW_DIR)
print('INTERIM_DIR:', INTERIM_DIR)
print('V1_DATA_DIR:', V1_DATA_DIR)
print('MOVIE_5171_DIR:', MOVIE_5171_DIR)
print('REPORTS_DIR:', REPORTS_DIR)
print('DATA_OUTPUT_DIR:', DATA_OUTPUT_DIR)
print('TABLES_DIR:', TABLES_DIR)


cwd: c:\Code\ott-churn-prediction\park.ingyeom\notebooks
PROJECT_ROOT: c:\Code\ott-churn-prediction
DATA_ROOT: c:\Code\ott-churn-prediction\_data
RAW_DIR: c:\Code\ott-churn-prediction\_data\01_raw
INTERIM_DIR: c:\Code\ott-churn-prediction\_data\02_interim
V1_DATA_DIR: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)
MOVIE_5171_DIR: c:\Code\ott-churn-prediction\_data\02_interim\260506_movie(5171)
REPORTS_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports
DATA_OUTPUT_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\03_movie_metadata_unification
TABLES_DIR: c:\Code\ott-churn-prediction\park.ingyeom\reports\tables\03_movie_metadata_unification


In [22]:
FILE_NAMES = {
    'movie_master': 'Movie_Master_v1.csv',
    'wavve': 'Wavve_movie(Regex).csv',
    'kobis': 'Wavve_movie(KOBIS).csv',
    'view': 'View_History_v1.csv',
    'movie_5171': 'movie_5171.csv',
}

SEARCH_DIRS = {
    'movie_master': [V1_DATA_DIR, INTERIM_DIR, RAW_DIR, DATA_ROOT, PROJECT_ROOT, Path('/mnt/data')],
    'view': [V1_DATA_DIR, INTERIM_DIR, RAW_DIR, DATA_ROOT, PROJECT_ROOT, Path('/mnt/data')],
    'wavve': [RAW_DIR, DATA_ROOT, PROJECT_ROOT, Path('/mnt/data')],
    'kobis': [RAW_DIR, DATA_ROOT, PROJECT_ROOT, Path('/mnt/data')],
    'movie_5171': [MOVIE_5171_DIR, INTERIM_DIR, DATA_ROOT, PROJECT_ROOT, Path('/mnt/data')],
}

def resolve_input_file(file_name: str, search_dirs: list[Path]) -> Path:
    candidates = [Path(d) / file_name for d in search_dirs]
    for p in candidates:
        if p.exists():
            return p
    checked = '\n'.join(str(p) for p in candidates)
    raise FileNotFoundError(f'Cannot find required input file: {file_name}\nChecked:\n{checked}')

def resolve_optional_input_file(file_name: str, search_dirs: list[Path]) -> Path | None:
    candidates = [Path(d) / file_name for d in search_dirs]
    for p in candidates:
        if p.exists():
            return p
    return None

PATH_MOVIE = resolve_input_file(FILE_NAMES['movie_master'], SEARCH_DIRS['movie_master'])
PATH_WAVVE = resolve_input_file(FILE_NAMES['wavve'], SEARCH_DIRS['wavve'])
PATH_KOBIS = resolve_input_file(FILE_NAMES['kobis'], SEARCH_DIRS['kobis'])
PATH_VIEW = resolve_input_file(FILE_NAMES['view'], SEARCH_DIRS['view'])
PATH_MOVIE_5171 = resolve_optional_input_file(FILE_NAMES['movie_5171'], SEARCH_DIRS['movie_5171'])

print('PATH_MOVIE:', PATH_MOVIE)
print('PATH_WAVVE:', PATH_WAVVE)
print('PATH_KOBIS:', PATH_KOBIS)
print('PATH_VIEW:', PATH_VIEW)
print('PATH_MOVIE_5171:', PATH_MOVIE_5171)


PATH_MOVIE: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\Movie_Master_v1.csv
PATH_WAVVE: c:\Code\ott-churn-prediction\_data\01_raw\Wavve_movie(Regex).csv
PATH_KOBIS: c:\Code\ott-churn-prediction\_data\01_raw\Wavve_movie(KOBIS).csv
PATH_VIEW: c:\Code\ott-churn-prediction\_data\02_interim\260430_membership_v1(이상치, 이름변경)\View_History_v1.csv
PATH_MOVIE_5171: c:\Code\ott-churn-prediction\_data\02_interim\260506_movie(5171)\movie_5171.csv


In [23]:
movie = pd.read_csv(PATH_MOVIE)
wavve = pd.read_csv(PATH_WAVVE)
kobis = pd.read_csv(PATH_KOBIS)
view = pd.read_csv(PATH_VIEW)
movie_5171 = pd.read_csv(PATH_MOVIE_5171) if PATH_MOVIE_5171 is not None else pd.DataFrame()

file_summary = pd.DataFrame([
    {'name': 'Movie_Master_v1', 'path': str(PATH_MOVIE), 'rows': len(movie), 'cols': movie.shape[1], 'required': True},
    {'name': 'Wavve_movie(Regex)', 'path': str(PATH_WAVVE), 'rows': len(wavve), 'cols': wavve.shape[1], 'required': True},
    {'name': 'Wavve_movie(KOBIS)', 'path': str(PATH_KOBIS), 'rows': len(kobis), 'cols': kobis.shape[1], 'required': True},
    {'name': 'View_History_v1', 'path': str(PATH_VIEW), 'rows': len(view), 'cols': view.shape[1], 'required': True},
    {'name': 'movie_5171', 'path': str(PATH_MOVIE_5171) if PATH_MOVIE_5171 is not None else '', 'rows': len(movie_5171), 'cols': movie_5171.shape[1] if not movie_5171.empty else 0, 'required': False},
])
file_summary.to_csv(TABLES_DIR / '03_movie_metadata_input_file_summary.csv', index=False, encoding='utf-8-sig')
file_summary


,name,path,rows,cols,required
0,Movie_Master_v1,c:\Code\ott-churn-prediction\_data\02_interim\...,14018,3,True
1,Wavve_movie(Regex),c:\Code\ott-churn-prediction\_data\01_raw\Wavv...,4060,31,True
2,Wavve_movie(KOBIS),c:\Code\ott-churn-prediction\_data\01_raw\Wavv...,999,13,True
3,View_History_v1,c:\Code\ott-churn-prediction\_data\02_interim\...,106205,5,True
4,movie_5171,c:\Code\ott-churn-prediction\_data\02_interim\...,5171,5,False


In [24]:
GENRE_MAP = {
    '드라마': '드라마',
    '액션': '액션',
    '스릴러': '스릴러/범죄',
    '범죄': '스릴러/범죄',
    '느와르': '스릴러/범죄',
    '미스터리': '스릴러/범죄',
    '코미디': '코미디',
    '로맨스': '로맨스',
    '멜로/로맨스': '로맨스',
    'SF': 'SF/판타지',
    '판타지': 'SF/판타지',
    'SF/판타지': 'SF/판타지',
    '모험': '모험/어드벤처',
    '어드벤처': '모험/어드벤처',
    '애니메이션': '애니메이션/키즈',
    '키즈': '애니메이션/키즈',
    '공포': '공포',
    '공포(호러)': '공포',
    '호러': '공포',
    '다큐멘터리': '다큐/교양',
    '교양': '다큐/교양',
    '가족': '가족',
    '전쟁': '전쟁/재난',
    '재난': '전쟁/재난',
    '전쟁/재난': '전쟁/재난',
    '음악': '기타',
    '공연': '기타',
    '스포츠': '기타',
    '서부': '기타',
    '서부극(웨스턴)': '기타',
    '사극': '기타',
    '무협': '기타',
    '성인물(에로)': '기타',
    '에로티시즘': '기타',
    '극장판': '기타',
    '단편': '기타',
    '뮤지컬': '기타',
    '기타': '기타',
}

GENRE_ORDER = [
    '드라마', '액션', '스릴러/범죄', '코미디', '로맨스', 'SF/판타지',
    '모험/어드벤처', '애니메이션/키즈', '공포', '다큐/교양', '가족',
    '전쟁/재난', '기타',
]

COUNTRY_MAP = {
    '대한민국': '한국',
    '한국': '한국',
    '미국': '미국',
    '일본': '일본',
    '중국': '중국',
    '홍콩': '홍콩',
    '영국': '영국',
    '프랑스': '프랑스',
    '독일': '독일',
    '캐나다': '캐나다',
    '러시아': '러시아',
    '대만': '대만',
    '호주': '호주',
    '이탈리아': '이탈리아',
    '스페인': '스페인',
    '기타': '기타',
}

In [25]:
def normalize_title(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).strip().lower()
    s = re.sub(r'[\[\]{}<>〈〉《》『』「」【】]', '', s)
    s = s.replace('（', '(').replace('）', ')')
    s = re.sub(r'\s+', '', s)
    s = re.sub(r"[-_:;,.!?'\"`~·ㆍ/\\|+*&^%$#@=]", '', s)
    return s

def clean_title_for_similarity(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value)
    s = re.sub(r'[\(\[\{（][^\)\]\}）]*[\)\]\}）]', '', s)
    return s.strip()

def normalize_title_no_year(value) -> str:
    return normalize_title(clean_title_for_similarity(value))

def extract_year_hint(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value)
    m = re.search(r'[\(\[\{（]\s*((?:19|20)\d{2})\s*[\)\]\}）]', s)
    return m.group(1) if m else ''

def sequence_similarity(a, b) -> float:
    a_norm = normalize_title_no_year(a)
    b_norm = normalize_title_no_year(b)
    if not a_norm or not b_norm:
        return 0.0
    return SequenceMatcher(None, a_norm, b_norm).ratio()

def split_multi(value) -> list[str]:
    if pd.isna(value):
        return []
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return []
    return [p.strip() for p in re.split(r'[|,;/]', s) if p and p.strip()]

def unique_join(values) -> str:
    seen = []
    for value in values:
        if pd.isna(value):
            continue
        s = str(value).strip()
        if not s or s.lower() == 'nan':
            continue
        if s not in seen:
            seen.append(s)
    return '|'.join(seen)

def normalize_genres(raw_values) -> list[str]:
    out = set()
    for raw in raw_values:
        for part in split_multi(raw):
            mapped = GENRE_MAP.get(part, GENRE_MAP.get(part.replace(' ', ''), '기타'))
            out.add(mapped)
    return [g for g in GENRE_ORDER if g in out]

def normalize_countries(raw_values) -> list[str]:
    out = []
    for raw in raw_values:
        for part in split_multi(raw):
            mapped = COUNTRY_MAP.get(part, part)
            if mapped not in out:
                out.append(mapped)
    return out

def normalize_age_rating_from_wavve(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return ''
    if s.endswith('.0'):
        s = s[:-2]
    if s in {'0', '전체', '전체관람가'}:
        return '전체'
    if s in {'7', '7세'}:
        return '7세'
    if s in {'12', '12세', '12세관람가', '12세이상관람가'}:
        return '12세'
    if s in {'15', '15세', '15세관람가', '15세이상관람가'}:
        return '15세'
    if s in {'18', '19', '청소년관람불가', '18세관람가'}:
        return '청불'
    return s

def normalize_age_rating_from_kobis(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return ''
    if '전체' in s or '모든 관람객' in s or '연소자관람가' in s:
        return '전체'
    if '12' in s:
        return '12세'
    if '15' in s or '중학생' in s:
        return '15세'
    if '청소년관람불가' in s or '18' in s or '연소자관람불가' in s or '미성년자관람불가' in s or '고등학생' in s:
        return '청불'
    return s

def open_year_from_open_dt(value) -> str:
    if pd.isna(value):
        return ''
    s = str(value).strip()
    digits = re.sub(r'\D', '', s)
    if len(digits) >= 4 and digits[:4].startswith(('19', '20')):
        return digits[:4]
    return ''

def parse_float(value):
    if pd.isna(value):
        return np.nan
    try:
        return float(value)
    except Exception:
        return np.nan

def parse_int(value):
    f = parse_float(value)
    if pd.isna(f):
        return np.nan
    return int(round(f))


def parse_movie_5171_runtime(value):
    """Parse movie_5171 showTM values such as '1시간 50분', '47분', or numeric minutes."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip()
    if not s or s.lower() == 'nan':
        return np.nan
    numeric = parse_float(s)
    if not pd.isna(numeric):
        return numeric
    hours = 0
    minutes = 0
    h_match = re.search(r'(\d+)\s*시간', s)
    m_match = re.search(r'(\d+)\s*분', s)
    if h_match:
        hours = int(h_match.group(1))
    if m_match:
        minutes = int(m_match.group(1))
    total = hours * 60 + minutes
    return float(total) if total > 0 else np.nan

def median_or_blank(values) -> str:
    vals = [v for v in values if not pd.isna(v)]
    if not vals:
        return ''
    return f'{float(np.median(vals)):.1f}'

def bool_int(condition) -> int:
    return int(bool(condition))

In [26]:
movie = movie.copy()
wavve = wavve.copy()
kobis = kobis.copy()
movie_5171 = movie_5171.copy()

movie['title_key_norm'] = movie['movie_title'].map(normalize_title)
wavve['title_key_norm'] = wavve['query_title'].map(normalize_title)
kobis['title_key_norm'] = kobis['wavve_title'].map(normalize_title)

if not movie_5171.empty:
    movie_5171['MOVIE_NUM'] = pd.to_numeric(movie_5171['MOVIE_ID'], errors='coerce').astype('Int64')
    movie_5171['title_key_norm'] = movie_5171['TITLE'].map(normalize_title)
else:
    movie_5171['MOVIE_NUM'] = pd.Series(dtype='Int64')
    movie_5171['title_key_norm'] = pd.Series(dtype='object')

key_summary = pd.DataFrame([
    {'table': 'Movie_Master', 'rows': len(movie), 'unique_title_key': movie['title_key_norm'].nunique(), 'duplicate_title_key_rows': int(movie['title_key_norm'].duplicated(keep=False).sum())},
    {'table': 'Wavve', 'rows': len(wavve), 'unique_title_key': wavve['title_key_norm'].nunique(), 'duplicate_title_key_rows': int(wavve['title_key_norm'].duplicated(keep=False).sum())},
    {'table': 'KOBIS', 'rows': len(kobis), 'unique_title_key': kobis['title_key_norm'].nunique(), 'duplicate_title_key_rows': int(kobis['title_key_norm'].duplicated(keep=False).sum())},
    {'table': 'movie_5171', 'rows': len(movie_5171), 'unique_title_key': movie_5171['title_key_norm'].nunique() if not movie_5171.empty else 0, 'duplicate_title_key_rows': int(movie_5171['title_key_norm'].duplicated(keep=False).sum()) if not movie_5171.empty else 0},
])
key_summary.to_csv(TABLES_DIR / '03_title_key_summary.csv', index=False, encoding='utf-8-sig')

movie_title_key_duplicates = (
    movie.loc[movie['title_key_norm'].duplicated(keep=False), ['MOVIE_NUM', 'movie_title', 'ott_release_month', 'title_key_norm']]
    .sort_values(['title_key_norm', 'MOVIE_NUM'])
    .copy()
)
movie_title_key_duplicates.to_csv(TABLES_DIR / '03_movie_master_title_key_duplicates.csv', index=False, encoding='utf-8-sig')

key_summary


,table,rows,unique_title_key,duplicate_title_key_rows
0,Movie_Master,14018,14012,12
1,Wavve,4060,3576,864
2,KOBIS,999,999,0
3,movie_5171,5171,5170,2


In [27]:
def aggregate_wavve(wavve_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for key, g in wavve_df.groupby('title_key_norm', dropna=False):
        if not key:
            continue
        genres_raw = []
        tags_raw = []
        countries_raw = []
        ages_norm = []
        runtimes = []
        years = []
        for _, r in g.iterrows():
            genres_raw.extend(split_multi(r.get('genre')))
            tags_raw.extend(split_multi(r.get('tags')))
            countries_raw.extend(split_multi(r.get('country')))
            age = normalize_age_rating_from_wavve(r.get('targetage'))
            if age:
                ages_norm.append(age)
            sec = parse_float(r.get('playtime_sec'))
            if not pd.isna(sec) and sec > 0:
                runtimes.append(sec / 60.0)
            year = parse_int(r.get('originalreleaseyear'))
            if not pd.isna(year):
                years.append(int(year))
        rows.append({
            'title_key_norm': key,
            'wavve_match_count': len(g),
            'wavve_movieids': unique_join(g.get('movieid', pd.Series(dtype=object)).tolist()),
            'wavve_titles': unique_join(g.get('title', pd.Series(dtype=object)).tolist()),
            'wavve_genre_raw': unique_join(genres_raw),
            'wavve_tags_raw': unique_join(tags_raw),
            'wavve_country_raw': unique_join(countries_raw),
            'wavve_age_rating_norm': unique_join(ages_norm),
            'wavve_runtime_min': median_or_blank(runtimes),
            'wavve_release_year': str(int(round(np.median(years)))) if years else '',
        })
    return pd.DataFrame(rows)

wavve_agg = aggregate_wavve(wavve)
wavve_agg.head()

,title_key_norm,wavve_match_count,wavve_movieids,wavve_titles,wavve_genre_raw,wavve_tags_raw,wavve_country_raw,wavve_age_rating_norm,wavve_runtime_min,wavve_release_year
0,007북경특급2,1,MV_AN01_AN0000000020,007 북경특급2,액션|드라마,액션|드라마,홍콩,15세,87.4,2014
1,100일동안100가지로100퍼센트행복찾기,1,MV_CV01_KE0000012213,100일 동안 100가지로 100퍼센트 행복찾기,코미디,코미디,독일,15세,111.0,2019
2,101마리의달마시안개(1961),1,MV_CA01_DY0000011204,(더빙) 101마리의 달마시안 개,애니메이션|모험,애니메이션|모험,미국,전체,79.3,1961
3,108영웅전설의무공,1,MV_ST01_ST000000839,108영웅: 전설의 무공,액션|무협,액션|무협,중국,15세,91.2,2017
4,10년,1,MV_CK01_TCO000012364,10년,드라마,드라마,일본,전체,99.3,2019


In [28]:
def aggregate_kobis(kobis_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for key, g in kobis_df.groupby('title_key_norm', dropna=False):
        if not key:
            continue
        genres_raw = []
        countries_raw = []
        ages_norm = []
        runtimes = []
        prdt_years = []
        open_years = []
        for _, r in g.iterrows():
            genres_raw.extend(split_multi(r.get('genreNm')))
            countries_raw.extend(split_multi(r.get('nationNm')))
            age = normalize_age_rating_from_kobis(r.get('watchGrade'))
            if age:
                ages_norm.append(age)
            runtime = parse_float(r.get('showTm'))
            if not pd.isna(runtime) and runtime > 0:
                runtimes.append(runtime)
            y = parse_int(r.get('prdtYear'))
            if not pd.isna(y):
                prdt_years.append(int(y))
            oy = open_year_from_open_dt(r.get('openDt'))
            if oy:
                open_years.append(oy)
        first = g.iloc[0]
        first_runtime = parse_float(first.get('showTm'))
        first_year = parse_int(first.get('prdtYear'))
        rows.append({
            'title_key_norm': key,
            'kobis_match_count': len(g),
            'kobis_movie_codes': unique_join(g.get('MovieCd', pd.Series(dtype=object)).tolist()),
            'kobis_titles': unique_join(g.get('movieNm(api)', pd.Series(dtype=object)).tolist()),
            'kobis_titles_en': unique_join(g.get('movieNmEn', pd.Series(dtype=object)).tolist()),
            'kobis_type_names': unique_join(g.get('typeNm', pd.Series(dtype=object)).tolist()),
            'kobis_directors': unique_join(g.get('directors', pd.Series(dtype=object)).tolist()),
            'kobis_actors': unique_join(g.get('actors', pd.Series(dtype=object)).tolist()),
            'kobis_wavve_titles': unique_join(g.get('wavve_title', pd.Series(dtype=object)).tolist()),
            'kobis_watch_grade_raw': unique_join(g.get('watchGrade', pd.Series(dtype=object)).tolist()),
            'kobis_genre_raw': unique_join(genres_raw),
            'kobis_country_raw': unique_join(countries_raw),
            'kobis_age_rating_norm': unique_join(ages_norm),
            'kobis_runtime_min': median_or_blank(runtimes),
            'kobis_release_year': str(int(round(np.median(prdt_years)))) if prdt_years else '',
            'kobis_open_year': unique_join(open_years),
            'kobis_selected_movie_code': first.get('MovieCd', ''),
            'kobis_selected_title': first.get('movieNm(api)', ''),
            'kobis_selected_title_en': first.get('movieNmEn', ''),
            'kobis_selected_type': first.get('typeNm', ''),
            'kobis_selected_directors': first.get('directors', ''),
            'kobis_selected_actors': first.get('actors', ''),
            'kobis_selected_watch_grade_raw': first.get('watchGrade', ''),
            'kobis_selected_genre_raw': first.get('genreNm', ''),
            'kobis_selected_country_raw': first.get('nationNm', ''),
            'kobis_selected_age_rating_norm': normalize_age_rating_from_kobis(first.get('watchGrade')),
            'kobis_selected_runtime_min': f'{float(first_runtime):.1f}' if not pd.isna(first_runtime) else '',
            'kobis_selected_release_year': str(int(first_year)) if not pd.isna(first_year) else '',
            'kobis_selected_open_year': open_year_from_open_dt(first.get('openDt')),
        })
    return pd.DataFrame(rows)

kobis_agg = aggregate_kobis(kobis)
kobis_agg.head()

,title_key_norm,kobis_match_count,kobis_movie_codes,kobis_titles,kobis_titles_en,kobis_type_names,kobis_directors,kobis_actors,kobis_wavve_titles,kobis_watch_grade_raw,...,kobis_selected_type,kobis_selected_directors,kobis_selected_actors,kobis_selected_watch_grade_raw,kobis_selected_genre_raw,kobis_selected_country_raw,kobis_selected_age_rating_norm,kobis_selected_runtime_min,kobis_selected_release_year,kobis_selected_open_year
0,007스카이폴,1,20113461,007 스카이폴,SKYFALL,장편,샘 멘데스,다니엘 크레이그|하비에르 바르뎀|주디 덴치|랄프 파인즈|나오미 해리스,007스카이폴,15세이상관람가,...,장편,샘 멘데스,다니엘 크레이그|하비에르 바르뎀|주디 덴치|랄프 파인즈|나오미 해리스,15세이상관람가,액션,미국|영국,15세,143.0,2011,2012
1,007스펙터,1,20157432,007 스펙터,Spectre,장편,샘 멘데스,다니엘 크레이그|레아 세이두|크리스토프 왈츠|모니카 벨루치,007스펙터,15세이상관람가,...,장편,샘 멘데스,다니엘 크레이그|레아 세이두|크리스토프 왈츠|모니카 벨루치,15세이상관람가,액션|어드벤처|범죄|스릴러,영국|미국,15세,147.0,2015,2015
2,007제로,1,2022A107,007 제로,Double zero,온라인전용,,,007제로,,...,온라인전용,NaN,NaN,NaN,액션,프랑스,,,2004,
3,10미니츠곤,1,20198121,10 미니츠 곤,10 MINUTES GONE,장편,브라이언 A 밀러,브루스 윌리스|마이클 치클리스,10미니츠곤,15세이상관람가,...,장편,브라이언 A 밀러,브루스 윌리스|마이클 치클리스,15세이상관람가,액션|범죄,캐나다|미국,15세,95.0,2019,2019
4,12디재스터,1,20142008,12 디재스터,12 Disasters,기타,스티븐 R. 몬로,에드 퀸|마그다 아파노위즈,12디재스터,,...,기타,스티븐 R. 몬로,에드 퀸|마그다 아파노위즈,NaN,SF,미국|캐나다,,,2012,


In [29]:
def aggregate_movie_5171(movie_5171_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if movie_5171_df.empty:
        return pd.DataFrame(columns=[
            'MOVIE_NUM', 'movie5171_match_count', 'movie5171_titles',
            'movie5171_genre_raw', 'movie5171_country_raw', 'movie5171_runtime_min'
        ])

    df = movie_5171_df.dropna(subset=['MOVIE_NUM']).copy()
    df['MOVIE_NUM'] = df['MOVIE_NUM'].astype(int)

    for movie_num, g in df.groupby('MOVIE_NUM', dropna=False):
        genres_raw = []
        countries_raw = []
        runtimes = []
        for _, r in g.iterrows():
            genres_raw.extend(split_multi(r.get('genre')))
            countries_raw.extend(split_multi(r.get('country')))
            runtime = parse_movie_5171_runtime(r.get('showTM'))
            if not pd.isna(runtime) and runtime > 0:
                runtimes.append(runtime)
        rows.append({
            'MOVIE_NUM': int(movie_num),
            'movie5171_match_count': len(g),
            'movie5171_titles': unique_join(g.get('TITLE', pd.Series(dtype=object)).tolist()),
            'movie5171_genre_raw': unique_join(genres_raw),
            'movie5171_country_raw': unique_join(countries_raw),
            'movie5171_runtime_min': median_or_blank(runtimes),
        })
    return pd.DataFrame(rows)

movie5171_agg = aggregate_movie_5171(movie_5171)

view_movie_nums = set(pd.to_numeric(view['MOVIE_NUM'], errors='coerce').dropna().astype(int).unique())
movie_master_nums = set(pd.to_numeric(movie['MOVIE_NUM'], errors='coerce').dropna().astype(int).unique())
movie5171_nums = set(movie5171_agg['MOVIE_NUM'].dropna().astype(int).unique()) if not movie5171_agg.empty else set()

movie5171_coverage_summary = pd.DataFrame([
    {'scope': 'Movie_Master_all', 'total_movies': len(movie_master_nums), 'movie5171_covered_movies': len(movie_master_nums & movie5171_nums), 'movie5171_missing_movies': len(movie_master_nums - movie5171_nums), 'movie5171_coverage_rate': len(movie_master_nums & movie5171_nums) / len(movie_master_nums) if movie_master_nums else 0},
    {'scope': 'View_History_movies_only', 'total_movies': len(view_movie_nums), 'movie5171_covered_movies': len(view_movie_nums & movie5171_nums), 'movie5171_missing_movies': len(view_movie_nums - movie5171_nums), 'movie5171_coverage_rate': len(view_movie_nums & movie5171_nums) / len(view_movie_nums) if view_movie_nums else 0},
])
movie5171_coverage_summary.to_csv(TABLES_DIR / '03_movie5171_coverage_summary.csv', index=False, encoding='utf-8-sig')

movie5171_agg.head()


,MOVIE_NUM,movie5171_match_count,movie5171_titles,movie5171_genre_raw,movie5171_country_raw,movie5171_runtime_min
0,3,1,그링고,코미디|액션|범죄|드라마|유럽에서 제작,미국|호주,111.0
1,5,1,국도극장,드라마,대한민국,106.0
2,6,1,맨인블랙2,SF|액션|코미디,미국,88.0
3,9,1,손오공:색즉시공,코미디|로맨스|드라마,대한민국,96.0
4,11,1,앵그리버드더무비,액션|코미디|가족|애니메이션|유럽에서 제작,미국|핀란드,97.0


In [30]:
def assess_kobis_quality(movie_title, row: pd.Series) -> dict:
    if row is None or row.empty:
        return {
            'title_year_hint': '',
            'kobis_title_similarity': '',
            'kobis_year_diff_min': '',
            'kobis_year_match_flag': '',
            'kobis_low_confidence_reasons': '',
            'kobis_quality_flag': '',
            'use_kobis_for_content_features': 0,
        }
    selected_title = row.get('kobis_selected_title', '') or row.get('kobis_titles', '')
    sim = sequence_similarity(movie_title, selected_title)
    year_hint = extract_year_hint(movie_title) or extract_year_hint(row.get('kobis_wavve_titles', ''))
    candidate_years = []
    for field in ['kobis_selected_release_year', 'kobis_selected_open_year', 'kobis_release_year', 'kobis_open_year']:
        for part in split_multi(row.get(field, '')):
            y = parse_int(part)
            if not pd.isna(y) and 1880 <= int(y) <= 2035:
                candidate_years.append(int(y))
    year_diff_min = ''
    year_match_flag = ''
    if year_hint:
        yh = int(year_hint)
        if candidate_years:
            diff = min(abs(yh - y) for y in candidate_years)
            year_diff_min = str(diff)
            year_match_flag = '1' if diff <= 1 else '0'
        else:
            year_match_flag = '0'
    reasons = []
    if sim < 0.60:
        reasons.append('low_title_similarity')
    if year_hint and year_match_flag == '0':
        reasons.append('year_hint_mismatch')
    if not str(row.get('kobis_selected_genre_raw', '')).strip():
        reasons.append('missing_genre')
    if not str(row.get('kobis_selected_country_raw', '')).strip():
        reasons.append('missing_country')
    if not str(row.get('kobis_selected_age_rating_norm', '')).strip():
        reasons.append('missing_age_rating')
    try:
        if int(row.get('kobis_match_count', 0)) > 1:
            reasons.append('multiple_kobis_rows_for_same_title')
    except Exception:
        pass
    if 'low_title_similarity' in reasons or 'year_hint_mismatch' in reasons or 'multiple_kobis_rows_for_same_title' in reasons:
        quality = 'D_kobis_low_confidence_excluded'
        use = 0
    elif year_hint:
        quality = 'B_kobis_high_confidence'
        use = 1
    else:
        quality = 'C_kobis_medium_confidence'
        use = 1 if sim >= 0.80 else 0
        if use == 0:
            quality = 'D_kobis_low_confidence_excluded'
            if 'medium_title_similarity_without_year_hint' not in reasons:
                reasons.append('medium_title_similarity_without_year_hint')
    return {
        'title_year_hint': year_hint,
        'kobis_title_similarity': f'{sim:.4f}' if selected_title else '',
        'kobis_year_diff_min': year_diff_min,
        'kobis_year_match_flag': year_match_flag,
        'kobis_low_confidence_reasons': '|'.join(reasons),
        'kobis_quality_flag': quality,
        'use_kobis_for_content_features': use,
    }

def make_content_flags(unified_genres, unified_countries, unified_age, unified_runtime, unified_release_year, wavve_tags_raw):
    runtime = parse_float(unified_runtime)
    year = parse_int(unified_release_year)
    is_kids_animation = ('애니메이션/키즈' in unified_genres) or ('키즈' in split_multi(wavve_tags_raw))
    is_family_content = ('가족' in unified_genres) or is_kids_animation or unified_age in {'전체', '7세', '12세'}
    return {
        'is_kids_animation': bool_int(is_kids_animation),
        'is_family_content': bool_int(is_family_content),
        'is_adult_content': bool_int(unified_age == '청불'),
        'is_korean_content': bool_int('한국' in unified_countries or '대한민국' in unified_countries),
        'is_us_content': bool_int('미국' in unified_countries),
        'is_japanese_content': bool_int('일본' in unified_countries),
        'is_recent_content': bool_int((not pd.isna(year)) and int(year) >= 2018),
        'is_old_content': bool_int((not pd.isna(year)) and int(year) <= 2010),
        'is_long_movie': bool_int((not pd.isna(runtime)) and runtime >= 120),
        'is_short_content': bool_int((not pd.isna(runtime)) and runtime < 60),
    }

In [31]:
base = (
    movie
    .merge(wavve_agg, on='title_key_norm', how='left')
    .merge(kobis_agg, on='title_key_norm', how='left')
    .merge(movie5171_agg, on='MOVIE_NUM', how='left')
)

# Fill selected audit columns as strings for stable downstream processing.
for col in base.columns:
    if col not in ['MOVIE_NUM', 'ott_release_month']:
        base[col] = base[col].fillna('')

base['source_wavve_flag'] = (base['wavve_match_count'].astype(str).str.len() > 0).astype(int)
base['source_kobis_flag'] = (base['kobis_match_count'].astype(str).str.len() > 0).astype(int)
base['source_movie5171_flag'] = (base['movie5171_match_count'].astype(str).str.len() > 0).astype(int)
base['primary_metadata_covered_flag'] = ((base['source_wavve_flag'] == 1) | (base['source_kobis_flag'] == 1)).astype(int)
base['metadata_covered_flag'] = ((base['source_wavve_flag'] == 1) | (base['source_kobis_flag'] == 1) | (base['source_movie5171_flag'] == 1)).astype(int)

def compose_metadata_source(row: pd.Series) -> str:
    parts = []
    if int(row['source_wavve_flag']) == 1:
        parts.append('wavve')
    if int(row['source_kobis_flag']) == 1:
        parts.append('kobis')
    if int(row['source_movie5171_flag']) == 1:
        parts.append('movie5171')
    return '+'.join(parts) if parts else 'missing'

base['metadata_source'] = base.apply(compose_metadata_source, axis=1)

quality_rows = []
for _, r in base.iterrows():
    quality_rows.append(assess_kobis_quality(r['movie_title'], r))
quality_df = pd.DataFrame(quality_rows)
base = pd.concat([base.reset_index(drop=True), quality_df.reset_index(drop=True)], axis=1)

base[['MOVIE_NUM', 'movie_title', 'metadata_source', 'source_wavve_flag', 'source_kobis_flag', 'source_movie5171_flag', 'metadata_covered_flag']].head()


,MOVIE_NUM,movie_title,metadata_source,source_wavve_flag,source_kobis_flag,source_movie5171_flag,metadata_covered_flag
0,0,걸어서하늘까지(1992),missing,0,0,0,0
1,1,너와극장에서,missing,0,0,0,0
2,2,가려진시간[가치봄],missing,0,0,0,0
3,3,그링고,wavve+movie5171,1,0,1,1
4,4,스위치(2010),missing,0,0,0,0


In [32]:
def build_unified_version(base_df: pd.DataFrame, version: str) -> pd.DataFrame:
    rows = []
    for _, r in base_df.iterrows():
        has_w = int(r['source_wavve_flag']) == 1
        has_k = int(r['source_kobis_flag']) == 1
        has_m = int(r.get('source_movie5171_flag', 0)) == 1

        use_kobis_values = False
        use_movie5171_values = False

        if version == 'v1':
            if has_w:
                use_for_content = 1
                metadata_quality = 'A_wavve'
            elif has_k:
                use_for_content = 1
                metadata_quality = 'B_or_C_kobis_unchecked'
                use_kobis_values = True
            elif has_m:
                use_for_content = 1
                metadata_quality = 'M_movie5171_auxiliary_fallback'
                use_movie5171_values = True
            else:
                use_for_content = 0
                metadata_quality = 'E_missing'
        elif version == 'v2':
            if has_w:
                use_for_content = 1
                metadata_quality = 'A_wavve'
            elif has_k and int(r['use_kobis_for_content_features']) == 1:
                use_for_content = 1
                metadata_quality = r['kobis_quality_flag']
                use_kobis_values = True
            elif has_m:
                use_for_content = 1
                metadata_quality = 'M_movie5171_auxiliary_fallback'
                use_movie5171_values = True
            elif has_k:
                use_for_content = 0
                metadata_quality = r['kobis_quality_flag'] or 'D_kobis_low_confidence_excluded'
            else:
                use_for_content = 0
                metadata_quality = 'E_missing'
        else:
            raise ValueError("version must be 'v1' or 'v2'")

        if has_w:
            raw_genre_values = [r.get('wavve_genre_raw', '')]
            raw_country_values = [r.get('wavve_country_raw', '')]
            unified_age = r.get('wavve_age_rating_norm', '') or r.get('kobis_age_rating_norm', '')
            unified_runtime = r.get('wavve_runtime_min', '') or r.get('kobis_runtime_min', '')
            unified_release_year = r.get('wavve_release_year', '') or r.get('kobis_release_year', '')
        elif use_kobis_values:
            raw_genre_values = [r.get('kobis_selected_genre_raw', '') or r.get('kobis_genre_raw', '')]
            raw_country_values = [r.get('kobis_selected_country_raw', '') or r.get('kobis_country_raw', '')]
            unified_age = r.get('kobis_selected_age_rating_norm', '') or r.get('kobis_age_rating_norm', '')
            unified_runtime = r.get('kobis_selected_runtime_min', '') or r.get('kobis_runtime_min', '')
            unified_release_year = r.get('kobis_selected_release_year', '') or r.get('kobis_release_year', '')
        elif use_movie5171_values:
            raw_genre_values = [r.get('movie5171_genre_raw', '')]
            raw_country_values = [r.get('movie5171_country_raw', '')]
            unified_age = ''
            unified_runtime = r.get('movie5171_runtime_min', '')
            unified_release_year = ''
        else:
            raw_genre_values = []
            raw_country_values = []
            unified_age = ''
            unified_runtime = ''
            unified_release_year = ''

        # movie_5171 is allowed to fill blank genre/country/runtime only. It never overwrites Wavve or trusted KOBIS values.
        used_movie5171_auxiliary = int(use_movie5171_values)
        if has_m and not raw_genre_values:
            raw_genre_values = [r.get('movie5171_genre_raw', '')]
            used_movie5171_auxiliary = 1 if r.get('movie5171_genre_raw', '') else used_movie5171_auxiliary
        if has_m and not raw_country_values:
            raw_country_values = [r.get('movie5171_country_raw', '')]
            used_movie5171_auxiliary = 1 if r.get('movie5171_country_raw', '') else used_movie5171_auxiliary
        if has_m and not str(unified_runtime).strip():
            unified_runtime = r.get('movie5171_runtime_min', '')
            used_movie5171_auxiliary = 1 if str(unified_runtime).strip() else used_movie5171_auxiliary

        unified_genres = normalize_genres(raw_genre_values)
        unified_countries = normalize_countries(raw_country_values)
        flags = make_content_flags(unified_genres, unified_countries, unified_age, unified_runtime, unified_release_year, r.get('wavve_tags_raw', ''))
        row = {
            'MOVIE_NUM': r['MOVIE_NUM'],
            'movie_title': r['movie_title'],
            'ott_release_month': r.get('ott_release_month', ''),
            'title_key_norm': r.get('title_key_norm', ''),
            'metadata_source': r.get('metadata_source', ''),
            'metadata_quality': metadata_quality,
            'use_for_content_features': use_for_content,
            'source_wavve_flag': int(r['source_wavve_flag']),
            'source_kobis_flag': int(r['source_kobis_flag']),
            'source_movie5171_flag': int(r.get('source_movie5171_flag', 0)),
            'used_movie5171_auxiliary_flag': int(used_movie5171_auxiliary),
            'primary_metadata_covered_flag': int(r.get('primary_metadata_covered_flag', 0)),
            'metadata_covered_flag': int(r['metadata_covered_flag']),
            'wavve_match_count': r.get('wavve_match_count', ''),
            'kobis_match_count': r.get('kobis_match_count', ''),
            'movie5171_match_count': r.get('movie5171_match_count', ''),
            'wavve_movieids': r.get('wavve_movieids', ''),
            'kobis_movie_codes': r.get('kobis_movie_codes', ''),
            'movie5171_titles': r.get('movie5171_titles', ''),
            'wavve_titles': r.get('wavve_titles', ''),
            'kobis_titles': r.get('kobis_titles', ''),
            'kobis_titles_en': r.get('kobis_titles_en', ''),
            'kobis_type_names': r.get('kobis_type_names', ''),
            'kobis_selected_movie_code': r.get('kobis_selected_movie_code', ''),
            'kobis_selected_title': r.get('kobis_selected_title', ''),
            'kobis_selected_title_en': r.get('kobis_selected_title_en', ''),
            'kobis_selected_type': r.get('kobis_selected_type', ''),
            'kobis_selected_directors': r.get('kobis_selected_directors', ''),
            'kobis_selected_actors': r.get('kobis_selected_actors', ''),
            'title_year_hint': r.get('title_year_hint', ''),
            'kobis_title_similarity': r.get('kobis_title_similarity', ''),
            'kobis_year_diff_min': r.get('kobis_year_diff_min', ''),
            'kobis_year_match_flag': r.get('kobis_year_match_flag', ''),
            'kobis_low_confidence_reasons': r.get('kobis_low_confidence_reasons', ''),
            'wavve_genre_raw': r.get('wavve_genre_raw', ''),
            'kobis_genre_raw': r.get('kobis_genre_raw', ''),
            'kobis_selected_genre_raw': r.get('kobis_selected_genre_raw', ''),
            'movie5171_genre_raw': r.get('movie5171_genre_raw', ''),
            'unified_genres': '|'.join(unified_genres),
            'wavve_tags_raw': r.get('wavve_tags_raw', ''),
            'wavve_country_raw': r.get('wavve_country_raw', ''),
            'kobis_country_raw': r.get('kobis_country_raw', ''),
            'kobis_selected_country_raw': r.get('kobis_selected_country_raw', ''),
            'movie5171_country_raw': r.get('movie5171_country_raw', ''),
            'unified_countries': '|'.join(unified_countries),
            'wavve_age_rating_norm': r.get('wavve_age_rating_norm', ''),
            'kobis_age_rating_norm': r.get('kobis_age_rating_norm', ''),
            'kobis_selected_age_rating_norm': r.get('kobis_selected_age_rating_norm', ''),
            'unified_age_rating': unified_age,
            'wavve_runtime_min': r.get('wavve_runtime_min', ''),
            'kobis_runtime_min': r.get('kobis_runtime_min', ''),
            'kobis_selected_runtime_min': r.get('kobis_selected_runtime_min', ''),
            'movie5171_runtime_min': r.get('movie5171_runtime_min', ''),
            'unified_runtime_min': unified_runtime,
            'wavve_release_year': r.get('wavve_release_year', ''),
            'kobis_release_year': r.get('kobis_release_year', ''),
            'kobis_open_year': r.get('kobis_open_year', ''),
            'kobis_selected_release_year': r.get('kobis_selected_release_year', ''),
            'kobis_selected_open_year': r.get('kobis_selected_open_year', ''),
            'unified_release_year': unified_release_year,
            **flags,
        }
        rows.append(row)
    return pd.DataFrame(rows)

unified_v1 = build_unified_version(base, 'v1')
unified_v2 = build_unified_version(base, 'v2')

unified_v1.shape, unified_v2.shape


((14018, 70), (14018, 70))

In [33]:
def coverage_by_scope(unified_df: pd.DataFrame, version: str) -> pd.DataFrame:
    view_movie_nums = set(view['MOVIE_NUM'].dropna().astype(int).unique())
    u = unified_df.copy()
    u['MOVIE_NUM_int'] = u['MOVIE_NUM'].astype(int)
    all_row = {
        'version': version,
        'scope': 'Movie_Master_all',
        'total_movies': len(u),
        'primary_metadata_covered': int(u['primary_metadata_covered_flag'].sum()),
        'movie5171_covered': int(u['source_movie5171_flag'].sum()),
        'metadata_covered': int(u['metadata_covered_flag'].sum()),
        'usable_for_content_features': int(u['use_for_content_features'].sum()),
        'primary_metadata_coverage_rate': float(u['primary_metadata_covered_flag'].mean()),
        'metadata_coverage_rate': float(u['metadata_covered_flag'].mean()),
        'usable_rate': float(u['use_for_content_features'].mean()),
    }
    uv = u[u['MOVIE_NUM_int'].isin(view_movie_nums)].copy()
    view_row = {
        'version': version,
        'scope': 'View_History_movies_only',
        'total_movies': len(uv),
        'primary_metadata_covered': int(uv['primary_metadata_covered_flag'].sum()),
        'movie5171_covered': int(uv['source_movie5171_flag'].sum()),
        'metadata_covered': int(uv['metadata_covered_flag'].sum()),
        'usable_for_content_features': int(uv['use_for_content_features'].sum()),
        'primary_metadata_coverage_rate': float(uv['primary_metadata_covered_flag'].mean()) if len(uv) else 0,
        'metadata_coverage_rate': float(uv['metadata_covered_flag'].mean()) if len(uv) else 0,
        'usable_rate': float(uv['use_for_content_features'].mean()) if len(uv) else 0,
    }
    return pd.DataFrame([all_row, view_row])

coverage = pd.concat([
    coverage_by_scope(unified_v1, 'v1'),
    coverage_by_scope(unified_v2, 'v2'),
], ignore_index=True)
coverage.to_csv(TABLES_DIR / '03_metadata_coverage_summary.csv', index=False, encoding='utf-8-sig')
coverage


,version,scope,total_movies,primary_metadata_covered,movie5171_covered,metadata_covered,usable_for_content_features,primary_metadata_coverage_rate,metadata_coverage_rate,usable_rate
0,v1,Movie_Master_all,14018,4578,5171,5173,5173,0.326580,0.369026,0.369026
1,v1,View_History_movies_only,5196,4576,5171,5171,5171,0.880677,0.995189,0.995189
2,v2,Movie_Master_all,14018,4578,5171,5173,5173,0.326580,0.369026,0.369026
3,v2,View_History_movies_only,5196,4576,5171,5171,5171,0.880677,0.995189,0.995189


In [34]:
quality_counts = pd.concat([
    unified_v1.assign(version='v1').groupby(['version', 'metadata_source', 'metadata_quality', 'use_for_content_features']).size().reset_index(name='count'),
    unified_v2.assign(version='v2').groupby(['version', 'metadata_source', 'metadata_quality', 'use_for_content_features']).size().reset_index(name='count'),
], ignore_index=True)
quality_counts.to_csv(TABLES_DIR / '03_metadata_quality_counts.csv', index=False, encoding='utf-8-sig')
quality_counts

,version,metadata_source,metadata_quality,use_for_content_features,count
0,v1,kobis+movie5171,B_or_C_kobis_unchecked,1,1000
1,v1,missing,E_missing,0,8845
2,v1,movie5171,M_movie5171_auxiliary_fallback,1,595
3,v1,wavve,A_wavve,1,2
4,v1,wavve+movie5171,A_wavve,1,3576
5,v2,kobis+movie5171,B_kobis_high_confidence,1,121
6,v2,kobis+movie5171,C_kobis_medium_confidence,1,852
7,v2,kobis+movie5171,M_movie5171_auxiliary_fallback,1,27
8,v2,missing,E_missing,0,8845
9,v2,movie5171,M_movie5171_auxiliary_fallback,1,595


In [35]:
low_confidence_kobis = unified_v2.loc[
    unified_v2['metadata_quality'].astype(str).str.startswith('D_'),
    [
        'MOVIE_NUM', 'movie_title', 'metadata_source', 'metadata_quality',
        'kobis_selected_title', 'title_year_hint', 'kobis_selected_release_year',
        'kobis_selected_open_year', 'kobis_title_similarity', 'kobis_low_confidence_reasons',
        'kobis_selected_genre_raw', 'kobis_selected_country_raw', 'kobis_selected_age_rating_norm'
    ]
].copy()
low_confidence_kobis.to_csv(TABLES_DIR / '03_low_confidence_kobis_rows.csv', index=False, encoding='utf-8-sig')
low_confidence_kobis.head(20)

,MOVIE_NUM,movie_title,metadata_source,metadata_quality,kobis_selected_title,title_year_hint,kobis_selected_release_year,kobis_selected_open_year,kobis_title_similarity,kobis_low_confidence_reasons,kobis_selected_genre_raw,kobis_selected_country_raw,kobis_selected_age_rating_norm


In [36]:
def explode_count(df: pd.DataFrame, col: str, version: str, top_n: int = 30) -> pd.DataFrame:
    counter = Counter()
    for value in df.loc[df['use_for_content_features'] == 1, col].fillna('').astype(str):
        for part in split_multi(value):
            counter[part] += 1
    return pd.DataFrame([
        {'version': version, 'field': col, 'value': k, 'count': v}
        for k, v in counter.most_common(top_n)
    ])

top_values = pd.concat([
    explode_count(unified_v2, 'unified_genres', 'v2'),
    explode_count(unified_v2, 'unified_countries', 'v2'),
    explode_count(unified_v2, 'unified_age_rating', 'v2'),
], ignore_index=True)
top_values.to_csv(TABLES_DIR / '03_top_unified_metadata_values_v2.csv', index=False, encoding='utf-8-sig')
top_values.head(40)

,version,field,value,count
0,v2,unified_genres,드라마,2385
1,v2,unified_genres,액션,1592
2,v2,unified_genres,스릴러,1431
3,v2,unified_genres,범죄,1431
4,v2,unified_genres,코미디,945
5,v2,unified_genres,SF,812
6,v2,unified_genres,판타지,812
7,v2,unified_genres,로맨스,756
8,v2,unified_genres,기타,508
9,v2,unified_genres,모험,488


In [37]:
OUT_V1 = OUTPUT_DIR / 'movie_metadata_unified_v1.csv'
OUT_V2 = OUTPUT_DIR / 'movie_metadata_unified_v2.csv'
OUT_SUMMARY_JSON = OUTPUT_DIR / 'movie_metadata_unified_v1_v2_summary.json'

unified_v1.to_csv(OUT_V1, index=False, encoding='utf-8-sig')
unified_v2.to_csv(OUT_V2, index=False, encoding='utf-8-sig')

summary = {
    'input_files': {
        'movie_master': str(PATH_MOVIE),
        'wavve': str(PATH_WAVVE),
        'kobis': str(PATH_KOBIS),
        'view_history': str(PATH_VIEW),
        'movie_5171': str(PATH_MOVIE_5171) if PATH_MOVIE_5171 is not None else '',
    },
    'output_files': {
        'movie_metadata_unified_v1': str(OUT_V1),
        'movie_metadata_unified_v2': str(OUT_V2),
        'summary_json': str(OUT_SUMMARY_JSON),
    },
    'input_rows': file_summary.to_dict('records'),
    'title_key_summary': key_summary.to_dict('records'),
    'movie5171_coverage_summary': movie5171_coverage_summary.to_dict('records'),
    'coverage_summary': coverage.to_dict('records'),
    'quality_counts': quality_counts.to_dict('records'),
    'low_confidence_kobis_count': int(len(low_confidence_kobis)),
    'recommended_downstream_file': str(OUT_V2),
    'movie5171_policy': 'auxiliary fallback only; does not overwrite Wavve or trusted KOBIS fields',
    'output_policy': 'derived analysis outputs are saved under park.ingyeom/reports/data, not under repository-level _data',
}
OUT_SUMMARY_JSON.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved:', OUT_V1)
print('Saved:', OUT_V2)
print('Saved:', OUT_SUMMARY_JSON)


Saved: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\03_movie_metadata_unification\movie_metadata_unified_v1.csv
Saved: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\03_movie_metadata_unification\movie_metadata_unified_v2.csv
Saved: c:\Code\ott-churn-prediction\park.ingyeom\reports\data\03_movie_metadata_unification\movie_metadata_unified_v1_v2_summary.json


In [38]:
assert len(unified_v1) == len(movie), 'v1 row count must match Movie_Master row count.'
assert len(unified_v2) == len(movie), 'v2 row count must match Movie_Master row count.'
assert unified_v2['MOVIE_NUM'].isna().sum() == 0, 'MOVIE_NUM should not be missing.'
assert unified_v2['movie_title'].isna().sum() == 0, 'movie_title should not be missing.'
assert int((unified_v2['metadata_quality'].astype(str).str.startswith('D_') & (unified_v2['use_for_content_features'] == 1)).sum()) == 0, 'Low-confidence KOBIS rows must not be used.'

final_check = pd.DataFrame([
    {'check': 'v1_rows_equal_movie_master', 'value': len(unified_v1) == len(movie)},
    {'check': 'v2_rows_equal_movie_master', 'value': len(unified_v2) == len(movie)},
    {'check': 'v2_recommended_for_downstream', 'value': True},
    {'check': 'v2_low_confidence_rows_excluded_from_content_features', 'value': int((unified_v2['metadata_quality'].astype(str).str.startswith('D_') & (unified_v2['use_for_content_features'] == 1)).sum()) == 0},
    {'check': 'movie5171_loaded_as_optional_auxiliary_source', 'value': PATH_MOVIE_5171 is not None},
    {'check': 'output_dir_under_reports_data_03', 'value': 'reports' in str(OUTPUT_DIR) and 'data' in str(OUTPUT_DIR) and '03_movie_metadata_unification' in str(OUTPUT_DIR)},
])
final_check.to_csv(TABLES_DIR / '03_movie_metadata_final_checks.csv', index=False, encoding='utf-8-sig')
final_check


,check,value
0,v1_rows_equal_movie_master,True
1,v2_rows_equal_movie_master,True
2,v2_recommended_for_downstream,True
3,v2_low_confidence_rows_excluded_from_content_f...,True
4,movie5171_loaded_as_optional_auxiliary_source,True
5,output_dir_under_reports_data_03,True


## 03번 결론

이 노트북은 `Movie_Master_v1.csv`를 기준으로 Wavve, KOBIS, movie_5171 보조 메타데이터를 통합한다.

다운스트림 분석에서는 `movie_metadata_unified_v2.csv`를 사용한다.

`v2`는 Wavve 메타데이터를 우선 사용하고, Wavve가 없는 경우에만 신뢰 가능한 KOBIS를 보완으로 사용한다. KOBIS는 제목 유사도와 제목 내 연도 힌트로 검증하며, 저신뢰 매칭은 `use_for_content_features=0`으로 제외한다.

`movie_5171.csv`는 `MOVIE_ID` 기준으로 직접 붙는 보조 소스다. 이 파일은 출처 검증 정보가 없으므로 Wavve 또는 신뢰 가능한 KOBIS 값을 덮어쓰지 않는다. 장르, 국가, 러닝타임이 비어 있는 경우에만 3순위 fallback으로 사용한다.

다음 단계인 `04_usage_feature_engineering.ipynb`는 02번에서 생성한 관측창 시청이력으로 구독 이벤트별 사용 행동 피처를 만든다. 이후 `05_content_feature_engineering.ipynb`에서 이 노트북의 `movie_metadata_unified_v2.csv`를 사용한다.
